In [1]:
# ==========================================
# APRIORI ALGORITHM
# WITH L1–L3 + STRONG RULE FILTER + L3 CHECK
# ==========================================

from itertools import combinations

#
# DATASET
#
transactions = [
    ["Milk", "Bread", "Butter"], # T1
    ["Bread", "Butter"],         # T2
    ["Milk", "Bread"],           # T3
    ["Milk", "Butter"],          # T4
    ["Bread", "Butter"]          # T5
]

n_transactions = len(transactions)
min_support_count = 2
min_support = min_support_count / n_transactions
min_confidence = 0.60

#
# SUPPORT FUNCTION
#
def get_support(itemset):
    return sum(1 for t in transactions if set(itemset).issubset(set(t)))

#
# APRIORI ALGORITHM
#
def apriori():
    items = sorted(set(item for t in transactions for item in t))
    L = []

    L1 = {}
    for item in items:
        sup = get_support([item])
        if sup >= min_support_count:
            L1[(item,)] = sup

    L.append(L1)
    k = 2

    while True:
        prev_level = list(L[-1].keys())
        candidates = set()

        # join step
        for i in range(len(prev_level)):
            for j in range(i + 1, len(prev_level)):
                union = tuple(sorted(set(prev_level[i]) | set(prev_level[j])))
                if len(union) == k:
                    candidates.add(union)

        current_level = {}
        for c in candidates:
            sup = get_support(c)
            if sup >= min_support_count:
                current_level[c] = sup

        if not current_level:
            break

        L.append(current_level)
        k += 1

    return L

#
# PRINT L1, L2, L3 WITH CHECK
#
def print_itemsets(L):
    print("\n=== FREQUENT ITEMSETS ===\n")
    for i, level in enumerate(L):
        print(f"L{i+1}:")
        for itemset, sup in level.items():
            print(f"{itemset} -> Support Count: {sup}, Support: {round(sup/n_transactions, 2)}")
        print()

    # L3 check
    if len(L) < 3:
        print("L3: None (No frequent 3-itemsets found)\n")
    else:
        print("L3 exists.\n")

#
# STRONG RULE GENERATION
#
def generate_strong_rules(L):
    rules = []
    for level in L[1:]: # start from L2
        for itemset, sup_count in level.items():
            for i in range(1, len(itemset)):
                for antecedent in combinations(itemset, i):
                    consequent = tuple(set(itemset) - set(antecedent))
                    sup = sup_count / n_transactions
                    conf = sup_count / get_support(antecedent)
                    lift = conf / (get_support(consequent) / n_transactions)

                    if sup >= min_support and conf >= min_confidence and lift > 1:
                        rules.append({
                            "rule": f"{antecedent} -> {consequent}",
                            "support": round(sup, 2),
                            "confidence": round(conf, 2),
                            "lift": round(lift, 2)
                        })
    return rules

#
# INTERPRETATION
#
def interpret(lift):
    if lift > 1:
        return "Strong positive association"
    elif lift == 1:
        return "Independent"
    else:
        return "Weak association"

#
# RUN PROGRAM
#
L = apriori()
print_itemsets(L)

rules = generate_strong_rules(L)
print("=== STRONG RULES ===\n")

if not rules:
    print("No strong rules found.\n")
else:
    for r in rules:
        print(f"Rule: {r['rule']}")
        print(f"Support: {r['support']}")
        print(f"Confidence: {r['confidence']}")
        print(f"Lift: {r['lift']}")
        print(f"Interpretation: {interpret(r['lift'])}\n")


=== FREQUENT ITEMSETS ===

L1:
('Bread',) -> Support Count: 4, Support: 0.8
('Butter',) -> Support Count: 4, Support: 0.8
('Milk',) -> Support Count: 3, Support: 0.6

L2:
('Butter', 'Milk') -> Support Count: 2, Support: 0.4
('Bread', 'Butter') -> Support Count: 3, Support: 0.6
('Bread', 'Milk') -> Support Count: 2, Support: 0.4

L3: None (No frequent 3-itemsets found)

=== STRONG RULES ===

No strong rules found.

